# RAPIDS SVC w/ Feature Engineering
We will train a Linear SVC model and selectively add (non-linear) feature engineered interaction features. We will use forward feature selection to discover which features improve CV score. We will use group k fold with year as group since the test data is different years than train data.

We will use [RAPIDS cuDF][1] for dataframe operations (using `%load_ext cudf.pandas` zero code change magic command which converts all pandas code into cuDF code) and RAPIDS cuML for SVC. Both will run quickly on GPU instead of CPU. Since the data is small it doesn't make too much difference, but if the data was large then we would see a big speed boost. Speed is important when we want to perform lots of experiments in search of ideas to improve our model!

Our simple model achieves `CV=0.893` and `LB=0.856` woohoo! Discussion about this notebook is [here][2]

[1]: https://rapids.ai/cudf-pandas/
[2]: https://www.kaggle.com/competitions/playground-series-s5e3/discussion/568268

In [ ]:
%load_ext cudf.pandas
import pandas as pd, numpy as np

train = pd.read_csv("/kaggle/input/playground-series-s5e3/train.csv")
train['year_group'] = train['id']//365 
print("Train shape:", train.shape )
train.head()

In [ ]:
test = pd.read_csv("/kaggle/input/playground-series-s5e3/test.csv")
print("Test shape:", test.shape )
test.head()

# Explore Train and Test Data

We will check two things with EDA:
* Verify train distribution is similar to test distribution
* Analyze target relationship to binned features

Regarding (1), if train and test distribution differ, then we will need to transform features so that they are similar and our model can generalize from train to test. Regarding (2), understanding how target relates to features will help us build better models and/or feature engineer.

From EDA below, we see that train distribution and test distribution are similar. This is good. We also see that there are strong correlations between features and targets.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
RMV = ['rainfall','id','bucket','year_group']
FEATURES = [c for c in list( train.columns ) if not c in RMV]
print(f"We have {len(FEATURES)} basic features:")
print( FEATURES )

In [ ]:
for c in FEATURES:

    # PLOT TRAIN DISTRIBUTION COMPARED WITH TEST DISTRIBUTION
    plt.figure(figsize=(12,3))
    plt.subplot(1,2,1)
    sns.distplot(train[c],label='train')
    sns.distplot(test[c],label='test')
    plt.legend()
    plt.title(f"{c}")    

    # PLOT TARGET RELATIONSHIP WITH BINNED NUMERIC FEATURES
    plt.subplot(1,2,2)
    train['bucket'], bin_edges = pd.cut(train[c], bins=10, retbins=True, labels=False)
    bucket_means = train.groupby('bucket')['rainfall'].mean()
    bin_midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
    plt.plot(bin_midpoints, bucket_means, marker='o', linestyle='-')
    plt.xlabel(f'{c} (Binned)')
    plt.ylabel('Mean Rainfall')
    plt.title(f'Mean Rainfall per {c} (train)')
    plt.xticks(bin_midpoints, rotation=45)
    plt.grid()
    
    plt.show()

# RAPIDS SVC w/ Forward Feature Selection
Using the speed up GPU we will iterate though a list of engineered features and train a new model with adding each one by one to our basic features. Whenever a new feature improves our CV score, we will keep it and continue. If the feature does not improve CV score, we will discard it.

Note the model SVC wants all features to have mean=0 and std=1, so we standardize all features inside each of our k fold loops. Since the data is small, we will cautiously avoid overfitting by using a linear SVC model (and we are selectively adding some non-linear engineered features).

In [ ]:
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import roc_auc_score
#from sklearn.svm import SVC, LinearSVC
from cuml.svm import SVC, LinearSVC

In [ ]:
INTERACT = []
for i,c1 in enumerate(FEATURES):
    for j,c2 in enumerate(FEATURES[i+1:]):
        n = f"{c1}_{c2}"
        train[n] = train[c1] * train[c2]
        test[n] = test[c1] * test[c2]
        INTERACT.append(n)
print(f"There are {len(INTERACT)} interaction features:")
print( INTERACT )

In [ ]:
ADD  = []
best_auc = 0
best_oof = None
best_pred = None

# FORWARD FEATURE SELECTION 
for k,col in enumerate(['baseline']+INTERACT):

    FOLDS = train.year_group.nunique()
    kf = GroupKFold(n_splits=FOLDS) 
    
    oof_svc = np.zeros(len(train))
    pred_svc = np.zeros(len(test))

    if col!='baseline': ADD.append(col)

    # GROUP K FOLD USING YEAR AS GROUP
    for i, (train_index, test_index) in enumerate(kf.split(train,groups=train.year_group)):
        #print("#"*25)
        #print(f"### Fold {i+1}")
        #print("#"*25)

        # TRAIN AND VALID DATA
        x_train = train.loc[train_index,FEATURES+ADD ].copy()
        y_train = train.loc[train_index,"rainfall"]
        x_valid = train.loc[test_index,FEATURES+ADD ].copy()
        y_valid = train.loc[test_index,"rainfall"]
        x_test = test[FEATURES+ADD ].copy()

        # SVC WANTS STANDARIZED FEATURES
        for c in FEATURES+ADD:
            m = x_train[c].mean()
            s = x_train[c].std()
            x_train[c] = (x_train[c]-m)/s
            x_valid[c] = (x_valid[c]-m)/s
            x_test[c] = (x_test[c]-m)/s
            x_test[c] = x_test[c].fillna(0)

        # TRAIN SVC MODEL
        #model = SVC(C=0.1, probability=True, kernel='poly', degree=1)
        model = LinearSVC(C=0.1, probability=True)
        model.fit(x_train.values, y_train.values)
    
        # INFER OOF
        oof_svc[test_index] = model.predict_proba(x_valid.values)[:,1]
        # INFER TEST
        pred_svc += model.predict_proba(x_test.values)[:,1]
    
    # COMPUTE AVERAGE TEST PREDS
    pred_svc /= FOLDS

    # COMPUTE CV VALIDATION AUC SCORE
    true = train.rainfall.values
    m = roc_auc_score(true, oof_svc)
    
    if m>best_auc:
        print(f"NEW BEST with {col} at {m}")
        best_auc = m
        best_oof = oof_svc.copy()
        best_pred = pred_svc.copy()
    else:
        print(f"Worse with {col} at {m}")
        ADD.remove(col)

In [ ]:
print(f"We achieved CV SVC AUC = {best_auc:.4f} adding {len(ADD)} interactions features:")
print( ADD )

# Create Submission CSV
We will make a submission CSV using our best SVC model with all the engineered features that improved our CV score!

In [ ]:
sub = pd.read_csv("/kaggle/input/playground-series-s5e3/sample_submission.csv")
sub.rainfall = best_pred
print("Submission shape:", sub.shape )
sub.to_csv(f"submission.csv",index=False)
sub.head()